# AML fraud/laundering detection: full GPU training

Runs the pipeline from https://github.com/Sumanasn/aml-tgn-fraud-detection on Kaggle's free GPU.

**Before running:** Settings (right panel) -> Accelerator: GPU T4x2 (or P100) -> Internet: On.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)

In [ ]:
!git clone https://github.com/Sumanasn/aml-tgn-fraud-detection.git
%cd aml-tgn-fraud-detection

In [ ]:
# Kaggle ships torch pre-built against its GPU/CUDA already -- don't reinstall
# torch itself (that's a common way to silently end up back on a CPU wheel).
# Only add the packages Kaggle's base image doesn't already have.
!pip install -q torch-geometric xgboost
import torch_geometric
print('pyg:', torch_geometric.__version__)

In [ ]:
!python -m src.data_pipeline.download_amlsim --dataset 20K_cycle200

## 1. XGBoost baseline (tabular + hand-engineered graph features)

In [ ]:
!python -m src.training.train_baseline

## 2. GraphSAGE GNN (10-layer, residual + JumpingKnowledge)

Depth is matched to the ~10-12 hop cycle length found in the fraud subgraph (see `src/models/gnn.py` docstring). GPU makes this fast enough to bump epochs/hidden size if you want to push accuracy further -- edit the call below.

In [ ]:
from src.training.train_gnn import main as train_gnn
train_gnn(epochs=300, patience=30)

## 3. TGN (memory + temporal attention embedding)

This is the one that's genuinely slow on CPU (~1-2 min/epoch locally at batch_size=200,
hundreds of tiny sequential batches). On a T4 this should run substantially faster --
bump `epochs` here now that GPU is available; 3 epochs was only a local correctness smoke test.

In [ ]:
from src.training.train_tgn import main as train_tgn
train_tgn(epochs=10, batch_size=200)

## 4. Time-to-detection: 3-way comparison

XGBoost / GraphSAGE GNN are re-scored on cumulative snapshots; TGN is replayed once, causally.

In [ ]:
from src.training.time_to_detection import main as time_to_detection
time_to_detection()

In [ ]:
from IPython.display import Image
Image(filename='results/time_to_detection.png')

## 5. Copy results + checkpoints to /kaggle/working for download

In [ ]:
import shutil
shutil.copytree('results', '/kaggle/working/results', dirs_exist_ok=True)
shutil.copytree('checkpoints', '/kaggle/working/checkpoints', dirs_exist_ok=True)
print('Copied. Download from the Output tab, or commit the checkpoints back to the repo yourself if you want them versioned.')